<a id='contents'></a>

# Data processing

This notebook contains scripts for processing GISAID data for analysis.

## Contents

- [Overview and table of contents](#contents)
- [Loading libraries and global variables](#global)
- [Comparinng GISAID and past data](#gisaidvpast)  
    - [Convert `xls` files to `csv`](#xls2csv)

**DELETE LATER: Include any other relevant links here. To link to somewhere in the notebook, you can include the following line of code in a Markdown cell right above where you want to make the link:**
`<a id='TAG'></a>`
**The TAG that you set as the id will be what you use when making the link in another Markdown cell:**
`[This link would go to the previous tag.](#TAG)`

<a id='global'></a>

## Libraries and global variables

In [1]:
# Libraries and packages

print('This notebook was prepared using:')

from importlib import reload

import sys, os, glob
from copy import deepcopy
print('python version %s' % sys.version)

import numpy as np
print('numpy version %s' % np.__version__)

import scipy as sp
import scipy.stats as st
print('scipy version %s' % sp.__version__)

import pandas as pd
print('pandas version %s' % pd.__version__)

import matplotlib
from matplotlib import cm
import matplotlib.pyplot as plt
print('matplotlib version %s' % matplotlib.__version__)


# Global variables

# Directories
GISAID_DIR = 'data/GISAID'

# Columns in GISAID isolate files
COL_T_COLLECT = 'Collection_Date'
COL_T_SUBMIT = 'Submission_Date'
COL_ID_GISAID = 'Isolate_Id'
COL_ID_HA = 'HA Segment_Id'
COL_ID_NA = 'NA Segment_Id'
COL_LOCATION = 'Location'

This notebook was prepared using:
python version 3.11.11 | packaged by conda-forge | (main, Dec  5 2024, 14:21:42) [Clang 18.1.8 ]
numpy version 1.26.4
scipy version 1.15.1
pandas version 2.2.2
matplotlib version 3.10.0


<a id='gisaidvpast'></a>

## Comparing GISAID data and previous data set

<a id='xls2csv'></a>

### Convert `xls` to `csv`

In [2]:
# Retrieve xls files, convert to csv and save

xlsfiles = ['.'.join(f.split('.')[:-1]) for f in glob.glob(os.path.join(GISAID_DIR, '*isolates.xls'))]

for f in xlsfiles:
    df = pd.read_excel(f + '.xls', 0, index_col=None)
    df.to_csv(f + '.csv', encoding='utf-8', index=False)

### Concatenate metadata data frames

In [3]:
# Iterate over types and proteins

types = ['H3N2', 'H1N1']
prots = ['HA']

for t in types:
    for p in prots:
        type_prot = t + '-' + p
        filelist = glob.glob(os.path.join(GISAID_DIR, '%s*isolates.csv' % (type_prot)))
        print('%s' % type_prot)
        print('merging:')
        for f in filelist:
            print('\t%s' % f)
        
        df_concat = pd.concat([pd.read_csv(f, low_memory=False) for f in filelist], ignore_index=True)
        df_concat.to_csv(os.path.join(GISAID_DIR, '%s-isolates-merged.csv.gz' % type_prot), index=False)
        print('saved as %s\n' % os.path.join(GISAID_DIR, '%s-isolates-merged.csv.gz' % type_prot))

H3N2-HA
merging:
	data/GISAID/H3N2-HA-2018-02-02-2019-08-01-gisaid_epiflu_isolates.csv
	data/GISAID/H3N2-HA-2016-03-02-2018-02-01-gisaid_epiflu_isolates.csv
	data/GISAID/H3N2-HA-2022-12-02-2023-07-01-gisaid_epiflu_isolates.csv
	data/GISAID/H3N2-HA-2019-08-02-2022-04-01-gisaid_epiflu_isolates.csv
	data/GISAID/H3N2-HA-2024-04-02-2024-10-01-gisaid_epiflu_isolates.csv
	data/GISAID/H3N2-HA-2022-04-02-2022-12-01-gisaid_epiflu_isolates.csv
	data/GISAID/H3N2-HA-2009-01-01-2016-03-01-gisaid_epiflu_isolates.csv
	data/GISAID/H3N2-HA-2023-07-02-2024-04-01-gisaid_epiflu_isolates.csv
saved as data/GISAID/H3N2-HA-isolates-merged.csv.gz

H1N1-HA
merging:
	data/GISAID/H1N1-HA-2019-02-02-2021-06-01-gisaid_epiflu_isolates.csv
	data/GISAID/H1N1-HA-2024-03-02-2024-10-01-gisaid_epiflu_isolates.csv
	data/GISAID/H1N1-HA-2023-08-02-2024-03-01-gisaid_epiflu_isolates.csv
	data/GISAID/H1N1-HA-2009-01-01-2016-01-01-gisaid_epiflu_isolates.csv
	data/GISAID/H1N1-HA-2016-01-02-2019-02-01-gisaid_epiflu_isolates.csv
	da

### Compare with Edwin's data

In [2]:
# Define variables
f_e = 'data/H3N2_HA_ID_DATE.csv'
f_g = 'data/GISAID/H3N2-HA-isolates-merged.csv.gz'
c_id = 'EPI_SET_ID'
c_date = 'DATE'
c_gid = COL_ID_HA

# Convert GISAID ID to Edwin ID
def g2e(tag):
    return tag.split('|')[0]

# Read in data
df_e = pd.read_csv(f_e, low_memory=False)
df_g = pd.read_csv(f_g, low_memory=False)

# Create temporary Epi_ID column in GISAID data
df_g[c_id]= [g2e(tag) for tag in df_g[c_gid]]

# # Only include data with years-months
# print('lengths before/after removing incomplete dates (no month):\nwhen\told\tnew\nbefore\t%d\t%d' % (len(df_e), len(df_g)))
# df_e = df_e[df_e[c_date].str.count('X')<4]
# df_g = df_g[df_g[COL_T_COLLECT].str.count('-')>=1]
# print('after\t%d\t%d' % (len(df_e), len(df_g)))

# # Only include data with years-months-days
# print('')
# print('lengths before/after removing incomplete dates (no day):\nwhen\told\tnew\nbefore\t%d\t%d' % (len(df_e), len(df_g)))
# df_e = df_e[~df_e[c_date].str.contains('XX')]
# df_g = df_g[df_g[COL_T_COLLECT].str.count('-')==2]
# print('after\t%d\t%d' % (len(df_e), len(df_g)))

# Only include data collected from 2009 onwards
print('')
print('lengths before/after removing pre-2009 sequences:\nwhen\told\tnew\nbefore\t%d\t%d' % (len(df_e), len(df_g)))
df_e['year'] = [float(y.split('-')[0]) for y in df_e[c_date]]
df_g['year'] = [float(y.split('-')[0]) for y in df_g[COL_T_COLLECT]]
df_e = df_e[df_e['year']>=2009]
df_g = df_g[df_g['year']>=2009]
print('after\t%d\t%d' % (len(df_e), len(df_g)))

# Only include data collected before 2019
print('')
print('lengths before/after removing post-2018 sequences:\nwhen\told\tnew\nbefore\t%d\t%d' % (len(df_e), len(df_g)))
df_e = df_e[df_e['year']<=2009]
df_g = df_g[df_g['year']<=2009]
print('after\t%d\t%d' % (len(df_e), len(df_g)))


lengths before/after removing pre-2009 sequences:
when	old	new
before	61848	144906
after	48927	143464

lengths before/after removing post-2018 sequences:
when	old	new
before	48927	143464
after	1854	1030


In [5]:
n_match_max = 100
n_matches = 0
unmatched = []

print('id\t\tedwin\t\tcollected\tsubmitted')
for df_iter, df_row in df_e.iterrows():
    df_match = df_g[df_g[c_id]==df_row[c_id]]
    if len(df_match)>1:
        print('WARNING! %s matches multiple rows' % df_row[c_id])
    if len(df_match)>0:
        n_matches += 1
        if n_matches <= n_match_max:
            print('\t'.join([df_row[c_id], df_row[c_date], df_match.iloc[0][COL_T_COLLECT], df_match.iloc[0][COL_T_SUBMIT]]))
    if len(df_match)==0: # and n_matches<n_match_max:
        # n_matches += 1
        # print('\t'.join([df_row[c_id], df_row[c_date], '', '']))
        unmatched.append(df_row[c_id])

print('')
print('previous data: %d sequences, %d matches (%.1f%%)' % (len(df_e), n_matches, 100*n_matches/len(df_e)))
print('new data: %d sequences, %d matches (%.1f%%)' % (len(df_g), n_matches, 100*n_matches/len(df_g)))

id		edwin		collected	submitted
EPI330577	2011-03-28	2011-03-28	2011-08-08
EPI274234	2009-04-30	2009-04-30	2010-08-04
EPI274609	2009-04-27	2009-04-27	2010-08-09
EPI274745	2009-04-26	2009-04-26	2010-08-09
EPI274641	2009-04-29	2009-04-29	2010-08-09
EPI274713	2009-04-28	2009-04-28	2010-08-09
EPI362350	2009-01-12	2009-01-12	2012-04-03
EPI273986	2009-04-28	2009-04-28	2010-08-04
EPI273378	2009-04-30	2009-04-30	2010-08-03
EPI274178	2009-04-30	2009-04-30	2010-08-04
EPI273954	2009-04-28	2009-04-28	2010-08-04
EPI271832	2009-05-04	2009-05-04	2010-07-20
EPI267086	2009-04-27	2009-04-27	2010-06-15
EPI274186	2009-04-30	2009-04-30	2010-08-04
EPI270703	2009-04-28	2009-04-28	2010-07-07
EPI274226	2009-04-30	2009-04-30	2010-08-04
EPI274210	2009-04-30	2009-04-30	2010-08-04
EPI244754	2009-02-03	2009-02-03	2010-03-03
EPI244751	2009-03-01	2009-03-01	2010-03-03
EPI244763	2009-02-11	2009-02-11	2010-03-03
EPI245691	2009-01-02	2009-01-02	2010-03-03
EPI245685	2009-01-05	2009-01-05	2010-03-03
EPI245676	2009-01-07	20

In [8]:
df_db = pd.read_csv('data/strains_h3n2_ha.txt', sep='\t')
not_in_db = []
db_names = []
db_counts = []

for u in unmatched:
    df_match = df_db[df_db['Accession Number']==u]
    if len(df_match)==0:
        # print('WARNING! %s does not exist in strains database' % u)
        not_in_db.append(u)
    elif len(df_match)>1:
        print('WARNING! %s multiple matches in strains database' % u)
    else:
        lab = df_match.iloc[0]['Submitting Lab.']
        if lab in db_names:
            db_counts[db_names.index(lab)] += 1
        else:
            db_names.append(lab)
            db_counts.append(1)

print('%d ids not in database' % len(not_in_db))
print('')
f = open('not-in-db.dat', 'w')
for epi_id in not_in_db:
    f.write('%s\n' % epi_id)
f.close()

db_names = np.array(db_names)
db_counts = np.array(db_counts)

cord = np.argsort(db_counts)[::-1]
print('counts\tdatabase')
for i in cord:
    print('%d\t%s' % (db_counts[i], db_names[i]))

171 ids not in database

counts	database
3171	public_health_agency_of_canada_(phac)
2684	other_database_import
2173	instituto_de_salud_carlos_iii
500	medical_university_vienna
431	national_institute_of_infectious_diseases_(niid)
413	robert_koch_institute
364	who_collaborating_centre_for_reference_and_research_on_influenza
307	norwegian_institute_of_public_health
194	insa_national_institute_of_health_portugal
168	institut_pasteur
137	national_institute_for_medical_research
134	national_virus_reference_laboratory
113	instituto_de_salud_publica_de_chile
109	centers_for_disease_control_and_prevention
105	hospital_universitari_vall_d'hebron
79	instituto_adolfo_lutz
64	oswaldo_cruz_institute___fiocruz___laboratory_of_respiratory_viruses_and_measles_(lvrs)
59	malbran
48	statens_serum_institute
45	national_laboratory_for_health_environment_and_food
44	instituto_nacional_enfermedades_infecciosas_c.g.malbran
34	istituto_superiore_di_sanit?
34	who_chinese_national_influenza_center
33	crick_worldw

## Figures for slides

In [20]:
import figures as f
reload(f)

f.plot_simulation_slides(seed=6)

In [32]:
import figures as f
reload(f)

for [file_name, fig_name] in [['Fig_S1/AUC_vs_f_T0_2015_H3N2_HA',    '2015-H3N2-HA']]:
    data = dict(auc_file='data/%s.csv' % file_name, plot_null=True, plot_model=True, hide_left=False, 
                fig_name='fig-AUC-%s-slides' % fig_name)
    f.plot_AUC_slides(**data)

    data['plot_model'] = False
    data['fig_name']   = 'fig-AUC-%s-null-slides' % fig_name
    f.plot_AUC_slides(**data)


for [file_name, fig_name] in [['Fig_S1/AUC_vs_f_T0_2015_H1N1pdm_HA', '2015-H1N1-HA'],
                              ['Fig_5/AUC_vs_f_T0_2015_H3N2_NA',     '2015-H3N2-NA'],
                              ['Fig_5/AUC_vs_f_T0_2015_H1N1pdm_NA',  '2015-H1N1-NA'] ]:
    
    data = dict(auc_file='data/%s.csv' % file_name, plot_null=True, plot_model=True, hide_left=True, 
                fig_name='fig-AUC-%s-slides' % fig_name)
    f.plot_AUC_slides(**data)

    data['plot_model'] = False
    data['fig_name']   = 'fig-AUC-%s-null-slides' % fig_name
    f.plot_AUC_slides(**data)

In [37]:
import figures as f
reload(f)

for [file_name, fig_name] in [['Fig_S2/Fix_prob_vs_f_T0_2015_H3N2_HA',    '2015-H3N2-HA'],
                              ['Fig_S2/Fix_prob_vs_f_T0_2015_H1N1pdm_HA', '2015-H1N1-HA']]:
    data = dict(pfix_file='data/%s.csv' % file_name, fig_name='fig-pfix-%s-slides' % fig_name)
    f.plot_pfix_slides(**data)